# frosty-analysis quickstart

Loads a `frosty-monitor` microSD card dump and produces the diagnostic
plots used to investigate a machine failure report. This notebook runs
against the small synthetic fixture checked into
`tests/fixtures/sample_deployment/` (see `tests/make_fixture.py`), so it
works standalone without a real field card.

The on-card format is defined in
[`../../docs/firmware/data-format-spec.md`](../../docs/firmware/data-format-spec.md).

In [ ]:
from pathlib import Path

from frosty_analysis import load_deployment, load_vibration_burst
from frosty_analysis.plots import (
    plot_burst_spectrogram,
    plot_burst_waveform,
    plot_channels,
    plot_state_timeline,
)

FIXTURE_DIR = Path("..") / "tests" / "fixtures" / "sample_deployment"

In [ ]:
dep = load_deployment(FIXTURE_DIR)

dep.manifest["deployment_id"], dep.manifest["machine_model"], len(dep.channels), len(dep.events)

## Channel time series

Beater/compressor current plus key cylinder and discharge-line
temperatures. Different-unit channels get their own small-multiple panel
rather than a misleading dual y-axis.

In [ ]:
fig, axes = plot_channels(dep)

## State timeline

The four boolean state channels (`beater_on`, `compressor_cmd`,
`tcc_satisfied`, `hp_ok`) as filled lanes, useful for spotting short-cycling
or a TCC that never satisfies at a glance.

In [ ]:
fig, ax = plot_state_timeline(dep)

## Vibration burst spectrogram

One captured burst decoded and plotted as a log-scaled spectrogram, plus
its raw 3-axis waveform.

In [ ]:
burst = load_vibration_burst(dep.burst_files[0])
burst.pod_id, burst.sample_rate_hz, burst.data.shape

In [ ]:
fig, ax = plot_burst_spectrogram(burst, axis="z")

In [ ]:
fig, ax = plot_burst_waveform(burst)

## Events

Button presses, system lifecycle messages, and automatic capture triggers
logged during the deployment.

In [ ]:
dep.events

## Failure signatures — roadmap

`frosty_analysis.signatures` has one stub function per documented failure
mode (short-cycling, TCC never satisfied, condenser airflow, knocking,
belt slip, motor degradation). Each currently raises `NotImplementedError`
— see `docs/roadmap.md` item 2 for the implementation plan. Docstrings on
each function describe the intended detection logic.

In [ ]:
from frosty_analysis import signatures

[name for name in dir(signatures) if name.startswith("detect_")]